# Seats of two connected flights and their bid prices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/revenue_management/notebooks/revenue_management.ipynb)

An airline based in Riga flies Barcelona → Riga and Riga → Helsinki on the same day, with a connection in Riga. It sells three trips (BCN–RIX, RIX–HEL and BCN–HEL with the connection), each with three fares. A BCN–HEL ticket needs a seat on both flights. Before sales open, the revenue manager decides how many tickets of each fare of each trip to put on sale. The notebook follows the decision in steps:

1. The flights, the trips and the fares.
2. A linear model of the two flights and its optimal plan.
3. The bid price of each flight and of each ticket, and the seats for which it holds.
4. An offer of a tour operator for connecting seats.
5. A change in the forecast demand of a fare.

**How to use it.** Run the cells in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders and drop-down lists): change the values and run the cell again. Outside Colab the form is a line such as `seats = 10`: edit the value and run the cell.

All the data and results are explained in the [document of the case](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/revenue_management/revenue_management.md). The data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup (run this cell first) { display-mode: "form" }
# In Google Colab, install the case from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/revenue_management"
    %pip install -q {url}
    # A session that already had the case keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "fares" or m.startswith("fares.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
import fares as fm

## 1. Flights, trips and fares

The first table gives the two flights and the seats of the aircraft of each. The second gives the nine fares of the trips: the trip, the fare, the flights the trip uses, the price of a one-way ticket (€/ticket), the forecast demand (tickets) and the seats one ticket uses on each flight of its trip. A Business ticket uses 1.5 seats, because Business uses the economy seats with the middle seat left free. The third table says what each fare includes.

If every forecast ticket were sold, how many seats would each flight need? Which fares of which trips compete for the seats of Riga → Helsinki?

In [ ]:
# @title Data { display-mode: "form" }
fm.show_data()

## 2. Linear model and optimal plan

$L$ is the set of flights, with index $l$; $T$ the set of trips, with index $t$ (BR = BCN–RIX, RH = RIX–HEL, BH = BCN–HEL); $F$ the set of fares, with index $f$ (Bus = Business, Flex = Economy FLEX, Basic = Economy BASIC); $T_l$ the trips that use flight $l$ ($T_{\text{BCN→RIX}}$ = {BR, BH}, $T_{\text{RIX→HEL}}$ = {RH, BH}). $r_{tf}$ is the price of a ticket of trip $t$ with fare $f$ (€/ticket), $D_{tf}$ its forecast demand (tickets), $s_f$ the seats a ticket with fare $f$ uses on each flight of its trip (1.5 for Business, 1 for the economy fares; seats/ticket) and $C_l$ the seats of the aircraft that flies flight $l$ (seats). The decision $x_{tf}$ is the number of tickets of trip $t$ with fare $f$ put on sale, a continuous variable.

$$
\begin{aligned}
\max \quad & \sum_{t \in T}\sum_{f \in F} r_{tf}\, x_{tf} && && \text{(revenue)} \\
\text{s.t.} \quad & \sum_{t \in T_l}\sum_{f \in F} s_f\, x_{tf} \le C_l && \forall l \in L && \text{(seats of the aircraft)} \\
& x_{tf} \le D_{tf} && \forall t \in T,\ f \in F && \text{(forecast demand)} \\
& x_{tf} \ge 0 && \forall t \in T,\ f \in F && \text{(non-negativity)}
\end{aligned}
$$

Written out term by term, the seat constraints of the two flights are:

$$
\begin{aligned}
l = \text{BCN→RIX}: \quad & 1.5\, x_{\text{BR,Bus}} + x_{\text{BR,Flex}} + x_{\text{BR,Basic}} + 1.5\, x_{\text{BH,Bus}} + x_{\text{BH,Flex}} + x_{\text{BH,Basic}} \le 220 \\
l = \text{RIX→HEL}: \quad & 1.5\, x_{\text{RH,Bus}} + x_{\text{RH,Flex}} + x_{\text{RH,Basic}} + 1.5\, x_{\text{BH,Bus}} + x_{\text{BH,Flex}} + x_{\text{BH,Basic}} \le 140
\end{aligned}
$$

The three BCN–HEL variables appear in both. The cell solves the model and gives the optimal plan: its revenue, the seats it uses on each flight and the tickets of each fare of each trip. The last table gives each constraint: what the plan uses, its right-hand side, whether it is **binding** (the plan uses all of it) and its **slack**.

Why does the plan put only 2 of the 80 BCN–HEL BASIC tickets on sale, although each one pays 110 €, more than a BCN–RIX BASIC ticket and more than a RIX–HEL BASIC ticket?

In [ ]:
# @title Optimal plan { display-mode: "form" }
fm.optimal_plan();

## 3. Bid prices

The **bid price** of a flight $l$, $y_l$, is the shadow price of the seat constraint of that flight: the rise in revenue with one more seat on that flight, in €/seat. It is the lowest revenue for which one more seat on that flight is worth selling. The first table gives the bid price of each flight. Choose a flight in the list: the cell solves the model again with one seat more and one seat less on it and prints the change in revenue. A table then gives the fares of the trips whose tickets change: their tickets with one seat less, in the optimal plan and with one seat more, and what each change brings with one seat more (tickets times price, in €). The changes add up to the bid price of the flight.

The bid price of a ticket of trip $t$ with fare $f$ is the bid prices of the flights of its trip times the seats it uses on each: $b_{tf} = s_f \sum_{l \in L_t} y_l$, where $L_t$ is the set of flights that trip $t$ uses. The third table gives it for the nine fares of the trips, next to the price, and the shadow price of the demand of each: the rise in revenue with one more ticket of forecast demand. The last table gives, for each flight, the seats of the aircraft for which its bid price holds: the model is solved again with the seats changed, and the bid price holds while the revenue changes by exactly the bid price for each seat.

No fare costs 35 €. Where does the bid price of Riga → Helsinki come from? Which tickets have a price equal to their bid price, and what does that say about their unsold demand? Beyond the upper end of its range, what does one more seat on a flight do?

In [ ]:
# @title Bid prices of the flights and of the tickets { display-mode: "form" }
flight = "RIX → HEL"  # @param ["BCN → RIX", "RIX → HEL"]
fm.bid_prices(flight)

## 4. Offer of a tour operator

A tour operator asks for BCN–HEL seats on top of the forecast demand. The bid prices estimate the change in revenue as the seats times the price offered minus the bid price of a BCN–HEL seat. The cell also solves the model again with the seats of the offer in the plan, and gives the fares of the trips whose tickets change. The figure draws, for 0 up to the seats of the offer, the estimate (red dashed line) and the change found by solving again (black points).

The sliders set the number of seats and the price per seat. For 120 €/seat, up to how many seats does the estimate agree with solving again? What happens to the bid price of Riga → Helsinki after the offer is accepted?

In [ ]:
# @title Offer of BCN–HEL seats { display-mode: "form" }
seats = 10  # @param {type:"slider", min:1, max:40, step:1}
price = 120  # @param {type:"slider", min:60, max:200, step:5}
fm.tour_operator_offer(seats, price);

## 5. Change in the forecast demand of a fare

The forecast demand is not fixed: a marketing campaign, a change of price or an agreement with an agency moves the demand of one fare of one trip, and the revenue manager puts those tickets on sale. What that is worth is the shadow price of the demand of that fare, and it holds while the new demand stays within a range. The first table gives both for the nine fares of the trips.

Choose a fare of a trip, the new forecast demand and what the change costs. The cell gives the shadow price of that demand and its range, the estimate with the shadow price, the change in revenue found by solving the model again, whether the new demand is inside the range, the bid prices of the flights with the new demand, the fares of the trips whose tickets change and the net result, with the estimate and solving again.

The campaign of the document raises RIX–HEL Economy FLEX from 40 to 50 tickets and costs 600 €: write that cost in the form. Would the estimate with the shadow price accept the campaign? Would solving again? Up to which new demand do the two agree?

In [ ]:
# @title Change in the forecast demand of a fare { display-mode: "form" }
product = "RIX–HEL Economy FLEX"  # @param ["BCN–RIX Business", "BCN–RIX Economy FLEX", "BCN–RIX Economy BASIC", "RIX–HEL Business", "RIX–HEL Economy FLEX", "RIX–HEL Economy BASIC", "BCN–HEL Business", "BCN–HEL Economy FLEX", "BCN–HEL Economy BASIC"]
new_demand = 50  # @param {type:"slider", min:0, max:200, step:1}
cost = 0  # @param {type:"number"}
fm.demand_change(product, new_demand, cost);

All the numbers are explained in the [document of the case](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/revenue_management/revenue_management.md), which also has a script that reproduces them.